# 02. Agentic RAG

> 검색을 항상 무조건 한 번만 돌리는 건 비효율적이에요. 관련성 체크 · 웹 폴백 · 쿼리 재작성으로 **에이전트 스스로 검색을 결정**하는 Agentic RAG를 만들어요.

이전 노트북(`01-Graph-Based-RAG.ipynb`)에서는 Naive RAG 그래프를 구현했어요. 이번 노트북에서는 RAG 파이프라인을 단계별로 고도화하는 방법을 배워요.

### 왜 Naive RAG로는 부족한가요?

Naive RAG는 항상 "검색 → 생성"을 실행해요. 하지만 실제 사용에서는 이런 문제가 생겨요:

| 상황 | Naive RAG의 문제 | 해결 방향 |
|------|-----------------|----------|
| PDF에 없는 최신 정보를 물어봄 | 무관한 문서로 엉뚱한 답변 생성 | 관련성 체크 + 웹 검색 폴백 |
| "서울 날씨"처럼 검색이 불필요한 질문 | 불필요한 검색 비용 발생 | 에이전트가 검색 여부 자율 판단 |
| "앤스로픽 투자"처럼 모호한 질문 | 벡터 검색 품질 저하 | 쿼리 재작성으로 검색어 최적화 |

> 🔑 **핵심 개념**: Naive RAG가 **자동판매기**라면, Agentic RAG는 **경험 많은 사서**예요. 자동판매기는 버튼을 누르면 무조건 음료를 주지만, 사서는 "이 질문에는 이 책이 필요하겠다", "이건 책 없이도 답할 수 있겠다"를 스스로 판단해요.

**이 노트북의 구성**

1. **관련성 체크(Groundedness Check)**: 검색 문서와 질문의 관련성을 평가해 분기 처리
2. **웹 검색 폴백(Web Search Fallback)**: 관련 문서가 없으면 Tavily로 웹 검색
3. **쿼리 재작성(Query Rewrite)**: 검색 전 LLM으로 질문을 최적화
4. **Agentic RAG**: 에이전트가 도구 사용 여부를 스스로 결정하는 완전한 에이전트 기반 시스템

## 학습 목표

이 노트북을 마치면 다음을 할 수 있어요:

1. `GroundednessChecker` 없이 LLM 기반 관련성 평가기를 직접 구현할 수 있어요
2. 조건부 엣지(Conditional Edges)로 관련성에 따라 흐름을 분기할 수 있어요
3. Tavily 웹 검색을 폴백 전략으로 연결해 무한 루프를 방지할 수 있어요
4. `create_retriever_tool`과 `ToolNode`를 사용해 완전한 Agentic RAG를 구축할 수 있어요

## 사전 지식

- `01-Graph-Based-RAG.ipynb` — Naive RAG 그래프, StateGraph 기초
- `05_Agent_Development/02-Tools-V1.ipynb` — Tool 정의와 ToolNode 사용법
- Pydantic BaseModel과 structured output 개념

## 전체 아키텍처

이번 노트북에서 구현할 Agentic RAG의 최종 형태예요. 각 단계를 순서대로 구현하면서 최종 아키텍처에 도달해요.

```mermaid
flowchart TD
    A([사용자 질문]) --> B[에이전트<br/>agent]
    B -- 도구 호출 --> C[PDF 검색<br/>retrieve]
    B -- 도구 불필요 --> Z([종료 END])
    C --> D{문서 관련성 평가<br/>grade_documents}
    D -- 관련 있음 --> E[답변 생성<br/>generate]
    D -- 관련 없음 --> F[쿼리 재작성<br/>rewrite]
    F --> B
    E --> Z

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef decision fill:#f8d7da,stroke:#dc3545,color:#721c24
    classDef storage fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e

    class A input
    class B,C,E,F process
    class D decision
    class Z output
```

| 구성 요소 | 역할 | 핵심 기술 |
|----------|------|----------|
| `agent` | 도구 사용 여부 결정 | `model.bind_tools()` + `tools_condition` |
| `retrieve` | PDF 문서 검색 | `create_retriever_tool` + `ToolNode` |
| `grade_documents` | 문서 관련성 평가 | `GradeDocuments` Pydantic + structured output |
| `rewrite` | 질문 최적화 | LLM 기반 쿼리 재작성 |
| `generate` | 최종 답변 생성 | RAG 프롬프트 + LLM |

> 💡 **핵심 정리**: Agentic RAG는 LLM이 수동적으로 검색 → 생성만 하는 것이 아니라, **언제 검색할지 스스로 결정**해요. 이것이 일반 RAG와 Agentic RAG의 핵심 차이예요.

## 1. 환경 설정

In [ ]:
# ---------------------------------------------------
# 환경 변수 로드
# ---------------------------------------------------
# .env 파일에 OPENAI_API_KEY, TAVILY_API_KEY 가 있어야 해요
from dotenv import load_dotenv

load_dotenv(override=True)

In [ ]:
# ---------------------------------------------------
# LangSmith 추적 설정 (선택)
# ---------------------------------------------------
# LangSmith 추적을 활성화하면 그래프 실행 과정을 시각적으로 디버깅할 수 있어요
import os

# os.environ["LANGCHAIN_TRACING_V2"] = "true"  # 추적 활성화
# os.environ["LANGCHAIN_PROJECT"] = "LangGraph-Agentic-RAG"  # 프로젝트 이름

## 2. RAG 유틸리티 — PDF 로더와 포맷터

이전 노트북에서 사용한 PDF 기반 Retrieval Chain을 그대로 활용해요.

> 🔑 **핵심 개념**: `pdf_retriever`와 `pdf_chain`을 분리해두면 LangGraph의 각 노드에서 독립적으로 활용할 수 있어요. 검색 단계와 생성 단계를 별도 노드로 설계할 수 있기 때문이에요.

In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: !uv add pdfplumber faiss-cpu
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
import os

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 기본 모델: gpt-4o-mini (비용 효율적)
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 3. 관련성 체크 — Groundedness Check

Naive RAG에서는 검색된 문서가 질문과 실제로 관련 있는지 확인하지 않고 바로 답변을 생성했어요. 관련성 체크(Groundedness Check)를 추가하면 검색 품질을 보장할 수 있어요.

```mermaid
flowchart LR
    A([질문 입력]) --> B[문서 검색<br/>retrieve]
    B --> C{관련성 체크<br/>relevance_check}
    C -- yes --> D[답변 생성<br/>llm_answer]
    C -- no --> B
    D --> E([답변 출력])

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef decision fill:#f8d7da,stroke:#dc3545,color:#721c24

    class A input
    class B,D process
    class C decision
    class E output
```

> 🔑 **핵심 개념**: 관련성 체크는 LLM에게 "이 문서가 질문에 답하는 데 도움이 되나요? yes/no"를 구조화된 출력으로 물어보는 방식이에요. `with_structured_output()`을 사용하면 Pydantic 모델로 타입 안전한 응답을 받을 수 있어요.

> ⚠️ **자주 하는 실수**: 관련성이 없을 때 단순히 `retrieve`로 되돌아가면 **무한 루프**가 발생해요. `recursion_limit`을 설정하고 `GraphRecursionError`를 잡는 것이 중요해요. 이 문제는 다음 섹션에서 웹 검색으로 해결해요.

In [ ]:
from typing import Annotated, TypedDict
from langgraph.graph.message import add_messages

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
from pydantic import BaseModel, Field
from langchain_core.prompts import PromptTemplate

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
from langgraph.graph import END, START, StateGraph
from langgraph.checkpoint.memory import MemorySaver

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 기본 엣지 연결
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
from IPython.display import Image, display

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 그래프 흐름: START → retrieve → relevance_check → (llm_answer | retrieve) → END
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
import uuid
from langchain_core.runnables import RunnableConfig
from langgraph.errors import GraphRecursionError

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 4. 웹 검색 폴백 — 무한 루프 해결

관련성이 없을 때 재검색으로 되돌아가면 무한 루프가 발생해요. 해결 방법은 관련성이 없을 때 **웹 검색(Web Search)**으로 폴백하는 거예요.

> 🔑 **핵심 개념**: 이전 섹션에서 `retrieve → relevance_check → retrieve` 루프가 영원히 반복되는 문제를 봤어요. 웹 검색 폴백은 이 루프의 **탈출구**를 만들어주는 거예요. 관련 문서가 없을 때 "같은 곳을 다시 뒤지는" 대신 "다른 서가(웹)를 찾아보는" 전략이에요.

```mermaid
flowchart LR
    A([질문 입력]) --> B[문서 검색<br/>retrieve]
    B --> C{관련성 체크}
    C -- yes --> D[답변 생성]
    C -- no --> E[웹 검색<br/>web_search]
    E --> D
    D --> F([답변 출력])

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef decision fill:#f8d7da,stroke:#dc3545,color:#721c24

    class A input
    class B,D,E process
    class C decision
    class F output
```

> 💡 **실무 팁**: `TavilySearch`를 사용하면 최신 정보(PDF에 없는 내용)도 답변할 수 있어요. `.env`에 `TAVILY_API_KEY`가 필요해요. Tavily API 키는 https://tavily.com 에서 무료로 발급받을 수 있어요.

> ⚠️ **자주 하는 실수**: 웹 검색 결과를 `context`에 저장할 때 기존 `llm_answer` 노드를 그대로 재사용할 수 있어요. 노드를 분리해 두면 입력 소스(PDF든 웹이든)에 관계없이 같은 답변 생성 로직을 쓸 수 있어요 — 이것이 모듈식 설계의 장점이에요.

In [ ]:
from langchain_tavily import TavilySearch

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 테스트: 2024년 노벨 문학상 수상자
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 그래프 흐름: START → retrieve → relevance_check → (llm_answer | web_search) → llm_answer → END
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 5. 쿼리 재작성 — Query Rewrite

사용자 질문은 대화체이거나 모호한 경우가 많아요. "앤스로픽 투자 금액"처럼 간략한 질문은 벡터 검색에서 관련 문서를 놓칠 수 있어요. 쿼리 재작성(Query Rewrite)을 추가하면 검색 품질을 향상시킬 수 있어요.

> 🔑 **핵심 개념**: `question` 필드를 `List[str]` + `add_messages`로 변경하면 원본 질문과 재작성된 질문을 누적 저장할 수 있어요. `state["question"][-1].content`로 항상 최신 질문을 가져올 수 있어요.

> 💡 **실무 팁**: 쿼리 재작성 프롬프트에 `[REMEMBER] Re-written question should be in the same language as the original question.`을 추가하면 한국어 질문이 영어로 재작성되는 것을 방지할 수 있어요.

In [ ]:
from typing import List

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
from langchain_core.output_parsers import StrOutputParser

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 그래프 흐름: START → query_rewrite → retrieve → relevance_check → (llm_answer | web_search) → llm_answer → END
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 6. Agentic RAG — 완전한 에이전트 기반 시스템

지금까지는 항상 PDF를 먼저 검색했어요. Agentic RAG에서는 LLM 에이전트가 **검색이 필요한지 스스로 판단**해요. PDF와 관련 없는 일반 상식 질문은 검색 없이 바로 답변해요.

> 💡 **핵심 정리**: `tools_condition`은 에이전트의 마지막 메시지에 `tool_calls`가 있는지 확인해요. 있으면 `tools` 노드로, 없으면 `END`로 라우팅해요. 이것이 에이전트 자율성의 핵심이에요.

> 🔑 **핵심 개념**: `create_retriever_tool`은 Retriever를 LLM이 호출할 수 있는 Tool로 변환해요. 에이전트는 질문에 따라 이 Tool을 호출할지 말지 스스로 결정해요.

> ⚠️ **자주 하는 실수**: Agentic RAG의 `AgentState`는 `messages` 리스트만 있어요. 이전 버전들의 `question`, `context`, `answer`가 없어요. 모든 정보가 메시지 체인 안에 포함되어 있어요.

In [ ]:
from langchain_core.tools.retriever import create_retriever_tool

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 에이전트가 언제 이 도구를 써야 하는지 description에 명확히 적어요
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
from langchain_core.messages import BaseMessage

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
from typing import Literal

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
from langchain_core.messages import HumanMessage

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 에이전트 LLM에 도구를 바인딩해요
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
from langgraph.prebuilt import ToolNode, tools_condition

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 에이전트 -> 도구 호출 여부에 따라 분기
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 그래프 흐름: START → agent → (retrieve | END) → grade_documents → (generate | rewrite) → END
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
from langgraph.errors import GraphRecursionError

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 7. TODO 실습 — 나만의 Agentic RAG 확장

아래 코드를 수정해서 Agentic RAG를 개선해봐요!

In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: Tavily 웹 검색을 두 번째 도구로 추가해봐요
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 핵심 요약

이 노트북에서 다음 내용을 배웠어요:

- **관련성 체크(Groundedness Check)**: `with_structured_output()`과 Pydantic 모델로 검색 문서의 관련성을 이진 평가해요
- **조건부 엣지(Conditional Edges)**: `add_conditional_edges()`와 라우팅 함수로 관련성에 따라 흐름을 분기해요
- **웹 검색 폴백**: `TavilySearch`를 폴백으로 추가해 무한 루프 없이 PDF에 없는 내용도 답변해요
- **쿼리 재작성(Query Rewrite)**: `question` 필드를 `List[str]`로 관리해 원본과 재작성 질문을 모두 보존해요
- **Agentic RAG**: `create_retriever_tool` + `ToolNode` + `tools_condition`으로 에이전트가 검색 여부를 스스로 결정해요
- **GradeDocuments**: Pydantic `BaseModel` + `with_structured_output()`으로 타입 안전한 문서 관련성 평가를 구현해요
- **GraphRecursionError**: `recursion_limit`과 예외 처리로 무한 루프를 방어해요

### 이 노트북에서 구현한 4가지 버전 비교

| 버전 | 핵심 추가 기능 | 무한 루프 방어 | LLM 호출 횟수 |
|------|--------------|---------------|--------------|
| **v1** 관련성 체크 | Groundedness Check | `recursion_limit`만 | 2-3회 (검색+평가+생성) |
| **v2** 웹 검색 폴백 | Tavily 웹 검색 | 웹 검색으로 탈출 | 3-4회 |
| **v3** 쿼리 재작성 | Query Rewrite | 웹 검색으로 탈출 | 4-5회 |
| **v4** Agentic RAG | 에이전트 자율 판단 | 재작성 + `recursion_limit` | 가변 (에이전트 판단) |

## 다음 노트북 예고

다음 `03-CRAG-Self-RAG.ipynb`에서는 **Corrective RAG(CRAG)와 Self-RAG**를 배워요. CRAG는 검색 결과를 평가하고 필요하면 쿼리를 수정 후 재검색하는 방식이에요. Self-RAG는 검색 여부, 검색 결과 활용 여부, 최종 답변의 환각 여부를 모두 LLM이 스스로 판단해요.